In [1]:
from fastapi import FastAPI
import pandas as pd
from pydantic import BaseModel, Field
import joblib
from fastapi.middleware.cors import CORSMiddleware



In [5]:
import sys
import sklearn
import numpy
import joblib

print("Python:", sys.version)
print("scikit-learn:", sklearn.__version__)
print("NumPy:", numpy.__version__)
print("Joblib:", joblib.__version__)

Python: 3.11.16 (main, Aug 14 2026, 15:34:35) [MSC v.1944 64 bit (AMD64)]
scikit-learn: 1.9.1
NumPy: 2.4.6
Joblib: 1.6.0


In [1]:
import sklearn

print(sklearn.__version__)

1.9.1


In [2]:
app = FastAPI()

In [3]:
@app.get('/')
def greet():
    return "Welcome to ML tutorial"


In [4]:
COLUMNS = ["latitude", "longitude", "price", "minimum_nights",
    "number_of_reviews", "reviews_per_month",
    "calculated_host_listings_count", "availability_365",
    "neighbourhood_group", "neighbourhood"]


In [5]:
model = joblib.load("ClassifierModel_Pipeline.pkl")

In [6]:
#Pydantic-> base model
class Features(BaseModel):
    latitude: float = Field(..., ge=-90, le=90, description="Latitude coordinate")
    longitude: float = Field(..., ge=-180, le=180, description="Longitude coordinate")
    price: float = Field(..., gt=0, description="Price per night, must be positive")
    minimum_nights: int = Field(..., ge=1, le=365, description="Minimum nights required for booking")
    number_of_reviews: int = Field(..., ge=0, description="Total number of reviews")
    reviews_per_month: float = Field(..., ge=0, description="Average reviews per month")
    calculated_host_listings_count: int = Field(..., ge=0, description="Number of listings by this host")
    availability_365: int = Field(..., ge=0, le=365, description="Days available out of 365")
    neighbourhood_group: str = Field(..., min_length=1, description="Borough or neighbourhood group")
    neighbourhood: str = Field(..., min_length=1, description="Specific neighbourhood name")



In [7]:
class PredictionResponse(BaseModel):
    Predicted_room_type: str
    Probability: list[float]

In [ ]:
@app.post("/predict")
def predict(features: Features):

    row = pd.DataFrame(
        [features.model_dump()],
        columns=COLUMNS
    )

    prediction = model.predict(row)
    probability = model.predict_proba(row)

    return {
        "Predicted_room_type": str(prediction.tolist()),
        "Probability": probability.tolist()[0]
    }

In [11]:
print([route.path for route in app.routes])

['/openapi.json', '/docs', '/docs/oauth2-redirect', '/redoc', '/', '/predict', '/predict']


In [11]:


nest_asyncio.apply()

import asyncio
import uvicorn

config = uvicorn.Config(
    app,
    host="127.0.0.1",
    port=2200
)

server = uvicorn.Server(config)


In [ ]:
await server.serve()

INFO:     Started server process [29068]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:2200 (Press CTRL+C to quit)


INFO:     127.0.0.1:61654 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:61654 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:65361 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:62956 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:62956 - "GET /openapi.json HTTP/1.1" 200 OK
